<a href="https://colab.research.google.com/github/GabrielvanderSchmidt/LangChain-Experiments/blob/main/retrieval/basic_ir_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Code strongly based on the LangChain docs and tutorials, more specifically:
#   - https://docs.langchain.com/oss/python/langchain/knowledge-base

In [ ]:
# @title Setup Env
!pip install -q transformers datasets accelerate bitsandbytes sentence-transformers chromadb \
                langchain langchain-huggingface langchain-chroma langchain-community langchain-ollama \
                --upgrade

# Basic IR with LangChain

In [11]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

from transformers import AutoTokenizer

import torch
import hashlib
import os

In [3]:
# @title Load `serafim-100m` embedding model

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device {device}.")
model_kwargs = {'device': device, # Use 'cuda:0' to specify a GPU, if multiple available
                # 'device_map': 'auto' # Use this to automatically spread the model across multiple GPUs, or
                #       to offload parts of the model from the GPU to the CPU if it's too big for VRAM. DON'T SET BOTH
                }

model_name="PORTULAN/serafim-100m-portuguese-pt-sentence-encoder"
embedding_model = HuggingFaceEmbeddings(model_name=model_name,
                                        model_kwargs=model_kwargs)
# Load tokenizer if using `from_huggingface_tokenizer` with `RecursiveCharacterTextSplitter`
embedding_tokenizer = AutoTokenizer.from_pretrained(model_name)
embedding_model

Using device cpu.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/171 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

HuggingFaceEmbeddings(model_name='PORTULAN/serafim-100m-portuguese-pt-sentence-encoder', cache_folder=None, model_kwargs={'device': 'cpu'}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [4]:
# @title Send model to another device, if needed

# We can offload a model to the CPU if not needed (e. g. after embedding is no longer necessary, or in order
# to make space for another model to be used)
embedding_model._client.to("cpu")

# And load it back to the original device
embedding_model._client.to(device)

SentenceTransformer(
  (0): Transformer({'max_seq_length': 128, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
)

In [5]:
# @title Check embedding max sequence length

# `sentence-transformers` sets models' max length to 128 by default (prioritizes embedding sentences/short text)
print(f"Current embedding model max sequence length: {embedding_model._client.max_seq_length}")

# We can change it if we want longer chunks (BERT-style models support a max sequence len of 512)
embedding_model._client.max_seq_length = 512

# Context windows larger than training (128) may lead to worse quality embeddings
# Short context windows might miss longer context clues
# Very large context windows may dilute minor context nuances

Current embedding model max sequence length: 128


In [12]:
# @title Load dummy document and split into chunks

# Download "database"
!curl -o glossary.md https://raw.githubusercontent.com/GabrielvanderSchmidt/SENAC-DLGlossary/refs/heads/main/glossary.md

# List of loaded documents
documents = [file for file in os.listdir(".") if os.path.isfile(file)]
documents_text = []
documents_meta = []
i = 0
for doc in documents:
    with open(doc, "r") as file:
        documents_text.append(file.read())
        documents_meta.append({"document_id" : str(i),
                               "tags" : ["learning", "AI"], # Other dummy metadata
                               "source" : doc})
        i = i+1

# Convert strings to Document objects
documents = [Document(page_content=text, metadata=meta) for text, meta in zip(documents_text, documents_meta)]


# There are two ways of instantiating the text splitter that I have found:

# 1) Just `RecursiveCharacterTextSplitter`, ignoring token counts
# Pros: Returns correct `start_index` consistently.
# Cons: Chunk size is given in number of characters, not tokens, so chunks are either smaller than what
#       fits inside the model (if chunk_size <= context_len) and could be bigger, or you risk having chunks
#       that don't fit inside the context length (if chunk_size > context_len).

# text_splitter = RecursiveCharacterTextSplitter(
#     chunk_size=512, # Chunk size (characters) -- we can't tell if it's smaller than embedder context length
#     chunk_overlap=200, # Chunk overlap (characters) so context isn't lost at the "edges" of chunks
#     add_start_index=True, # Track index in original document
# )

# 2) `RecursiveCharacterTextSplitter` with HF tokenizer
# Pros: Chunk size can be perfectly controled in terms of tokens, which enables us to make it as big as
#       possible without surpassing the embedder context length.
# Cons: At the time of testing, is often unable to track start_index, many chunks having a start_index
#       value of -1.

text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer=embedding_tokenizer, # Load tokenizer to calculate chunk length
    chunk_size=400, # Chunk size (tokens) -- must be smaller than embedder sequence length
    chunk_overlap=30, # Chunk overlap (tokens) so context isn't lost at the "edges" of chunks
    add_start_index=True, # (Try to) Track index in original document
)

# Apparently there is a solution being worked on (https://github.com/langchain-ai/langchain/issues/29884), so
# method 2 (or using `TokenTextSplitter.from_huggingface_tokenizer`) should (?) be the prefered alternative in the future

all_splits = text_splitter.split_documents(documents)
print(f"Documents split in {len(all_splits)} chunks.")

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 13632  100 13632    0     0  60722      0 --:--:-- --:--:-- --:--:-- 60857
Documents split in 15 chunks.


In [13]:
all_splits[0]

Document(metadata={'document_id': '0', 'tags': ['learning', 'AI'], 'source': 'glossary.md', 'start_index': 0}, page_content='# Estrutura da Rede\n\n\n#### Perceptron (Neurônio Artificial)\nÉ o tipo mais simples de rede neural **feedforward**, e hoje é usado como sinônimo para **neurônio artificial**, que é o bloco fundamental de **redes neurais artificiais**. Neurônios artificiais são agrupados lateralmente em **camadas densas**, e estas são empilhadas verticalmente para formar **redes neurais multicamadas**. Consiste de uma soma das entradas ponderadas pelos **pesos/parâmetros** conectados $$u = w_0 \\cdot X_0 + … + w_n \\cdot X_n + b$$ seguida de uma **função de ativação** não-linear $$g(u)$$. Um neurônio artificial é equivalente a uma regressão logística quando usando **função de ativação sigmóide**.')

In [14]:
# @title Instantiate and populate vector DB

vector_store = Chroma(
    collection_name="my_collection",
    embedding_function=embedding_model, # Embeddings are calculated on the fly
    persist_directory="./chroma_langchain_db", # Save data locally, or use host=[...] to connect to an existing ChromaDB server
    # host="localhost",
)

# Ids can be generated beforehand, or created by the `add_documents` call
# Let's generate them with our hashing function
def generate_ids(chunks):
    text_chunk = [chunk.page_content for chunk in chunks]
    encoded_chunks = [chunk.encode("utf-8") for chunk in text_chunk]
    hashes = [hashlib.md5(chunk).hexdigest() for chunk in encoded_chunks]
    return hashes

ids = generate_ids(all_splits)
vector_store.add_documents(documents=all_splits, ids=ids) # Outputs added ids

# Use this to update existing documents
# vector_store.update_documents(ids=ids, documents=all_splits)

# Adding/Updating documents takes some time, since data is being embedded on the CPU

['a53082a11a83531cf08d3f139d2f0fc7',
 '23d6edf23064b78701fbf0a32ef1d061',
 '36d2e65a3801f8aa4f871eda43ebb189',
 'c8442b47a8669e4a1e19d8456212ab2d',
 '42c106a0b3d5cc5c4caff7184f2306b7',
 '3457355bbd982b2efe9ece7124d39820',
 'da18a6e5616bc9bd7cc29ba5d67507f8',
 '656b109891dd3b9b615a409c5034c9eb',
 'd21123857fd180ae0e34c6ad768e2627',
 '1b33a5155c5632e893fceb3d3915ae25',
 '7e8a4b6c8894f8974021805d5d81f26d',
 '312788b84e467c9160c345cd2ec17398',
 '8ad5b46bc8feeb81bc85a0094dc76cab',
 '89e0938d1509a689ac7d98d950e00e4a',
 '69d4587b00a54624ef853a602577c815']

In [18]:
# @title Query DB

query = "Qual a diferença entre autoencoders e U-Nets?"
# Embed and search with text query
matches = vector_store.similarity_search_with_relevance_scores(query=query,
                                                               k=3 # How many matches to return
                                                               )

# Show top matches
print(f"Query: {query}")
for doc, score in matches[:2]:
    print(f"Similarity score: {score}\nDocument content: {doc.page_content}")
# ChromaDB scores seem to be unbound due to using non-normalized Euclidean distance as similarity metric

Query: Qual a diferença entre autoencoders e U-Nets?
Similarity score: -63.63533331178181
Document content: #### U-Net
Um tipo de **CNN** onde a rede segue um formato de "ampulheta", onde os **feature maps** primeiro reduzem em dimensão (altura x largura) até culminarem em um **gargalo** central, após o qual os feature maps voltam a aumentar em tamanho, geralmente até retornarem ao tamanho original da imagem de entrada. A diferença estrutural principal entre U-Nets e **CNN autoencoders** é que U-Nets possuem **skip connections** conectando níveis hierárquicos correspondentes nos dois lados da "ampulheta" (ex.: Bloco de entrada conectado com bloco de saída; primeiro bloco antes do gargalo com o primeiro bloco após o gargalo; etc).
Similarity score: -65.06914617023683
Document content: #### Autoencoder
Nome dado a redes seguindo um esquema de **aprendizado não supervisionado** onde o modelo deve aproximar uma **função identidade** ($$f(X)=X$$) apesar do fluxo de informação passar por um 

/tmp/ipykernel_959/2668463236.py:5: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='7e8a4b6c8894f8974021805d5d81f26d', metadata={'start_index': 8336, 'document_id': '0', 'tags': ['learning', 'AI'], 'source': 'glossary.md'}, page_content='#### U-Net\nUm tipo de **CNN** onde a rede segue um formato de "ampulheta", onde os **feature maps** primeiro reduzem em dimensão (altura x largura) até culminarem em um **gargalo** central, após o qual os feature maps voltam a aumentar em tamanho, geralmente até retornarem ao tamanho original da imagem de entrada. A diferença estrutural principal entre U-Nets e **CNN autoencoders** é que U-Nets possuem **skip connections** conectando níveis hierárquicos correspondentes nos dois lados da "ampulheta" (ex.: Bloco de entrada conectado com bloco de saída; primeiro bloco antes do gargalo com o primeiro bloco após o gargalo; etc).'), -63.63533331178181), (Document(id='312788b84e467c9160c345cd2ec17398', metadata={'tags': ['learning',

In [19]:
# @title Other ways to query DB

# If making multiple searches with the same query, you can embed the query once and search with its vector representation
query_vector = embedding_model.embed_query(query)

# Search by similarity
similarity_matches = vector_store.similarity_search_by_vector(embedding=query_vector,
                                                              k=3)
# Search by similarity AND output diversity
maxmarginal_matches = vector_store.max_marginal_relevance_search_by_vector(embedding=query_vector,
                                                                           k=3, # Number of docs in final selecion
                                                                           fetch_k=5, # Number of docs to look for "similar but diverse" matches
                                                                           lambda_mult=0.5, # 0 == maximum diversity, 1 == minimum diversity
                                                                           )

print(f"Top similarity match: {similarity_matches[0].page_content}\n")
print(f"Top MMR match: {maxmarginal_matches[0].page_content}")

Top similarity match: #### U-Net
Um tipo de **CNN** onde a rede segue um formato de "ampulheta", onde os **feature maps** primeiro reduzem em dimensão (altura x largura) até culminarem em um **gargalo** central, após o qual os feature maps voltam a aumentar em tamanho, geralmente até retornarem ao tamanho original da imagem de entrada. A diferença estrutural principal entre U-Nets e **CNN autoencoders** é que U-Nets possuem **skip connections** conectando níveis hierárquicos correspondentes nos dois lados da "ampulheta" (ex.: Bloco de entrada conectado com bloco de saída; primeiro bloco antes do gargalo com o primeiro bloco após o gargalo; etc).

Top MMR match: #### U-Net
Um tipo de **CNN** onde a rede segue um formato de "ampulheta", onde os **feature maps** primeiro reduzem em dimensão (altura x largura) até culminarem em um **gargalo** central, após o qual os feature maps voltam a aumentar em tamanho, geralmente até retornarem ao tamanho original da imagem de entrada. A diferença es

In [21]:
# @title Setup retrieval pipeline


# Set vector DB + embedding model as a retriever
base_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3} # Since we are working on a tiny database, retrieve only a few chunks
)

# Batch-search multiple queries
queries = [
        "Qual é o componente mais fundamental de redes neurais?",
        "Qual é o papel da função de ativação em uma rede neural?",
        "Como redes neurais aprendem?",
    ]
results = base_retriever.batch(queries)

for i, result in enumerate(results):
    print(f"Query: {queries[i]}")
    print(f"Result: \n{result[0].page_content}")
    print("\n\n")

Query: Qual é o componente mais fundamental de redes neurais?
Result: 
#### Camada Densa
É o tipo de camada neural mais simples encontrado dentro de uma **rede neural artificial**. É composta por vários **neurônios artificiais** que operam em conjunto, em um mesmo nível hierárquico dentro do fluxo de informação dentro da rede. É chamada de camada “densa” por todos os seus neurônios estarem conectados a todas as saídas da camada anterior, propriedade a qual pode levar ao aumento exponencial de **parâmetros** da rede se não observada durante a sua definição. Embora comumente apresentada como uma única unidade, englobando tanto a soma ponderada quanto a **função de ativação**, sua implementação em código frequentemente separa esta camada em uma transformação linear (ex.: `nn.Linear`) seguida de uma ativação (ex.: `nn.ReLU`).



Query: Qual é o papel da função de ativação em uma rede neural?
Result: 
# Estrutura da Rede


#### Perceptron (Neurônio Artificial)
É o tipo mais simples de rede 